In [0]:
# ============================================================
# CÉLULA 1 — Instalação de dependências
# Instala o SHAP, necessário para interpretar o modelo.
# Deve ser a PRIMEIRA célula do notebook.
# ============================================================
%pip install shap --quiet

In [0]:
# ============================================================
# CÉLULA 2 — Reiniciar a sessão Python
# Obrigatório após o %pip install: sem isso, o Python continua
# com os módulos antigos em memória e o import falha.
# ATENÇÃO: limpa tudo que estava em memória. Rode antes dos imports.
# ============================================================
dbutils.library.restartPython()

In [0]:
# ============================================================
# CÉLULA 3 — Imports e configuração
# Carrega as bibliotecas de modelagem, avaliação e interpretação.
# Versão SEM XGBoost: usa Random Forest, que já vem no sklearn.
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, ConfusionMatrixDisplay)
import mlflow

CATALOGO = "dbw_fiap_techchallenge_prod_7405607602465611"
TABELA = f"{CATALOGO}.default.fase3_features_contexto"

In [0]:
# ============================================================
# CÉLULA 4 — Matriz de features
# Define as 11 features do modelo final.
# Fora da matriz: metas futuras (redundantes), esforco_vs_uf
# (colinear com posicao_relativa_uf) e as colunas de vazamento.
# ============================================================
FEATURES = [
    "qtd_redes",
    "percentual_participacao",
    "meta_do_ano",
    "taxa_2023",
    "media_portugues_2023",
    "participacao_2023",
    "gap_meta_2023",
    "esforco_necessario",
    "taxa_uf_2023",
    "posicao_relativa_uf",
    "meta_vs_uf",
]
TARGET = "target"
GRUPO = "id_municipio"

pdf = spark.table(TABELA).toPandas()

X = pdf[FEATURES].copy()
y = pdf[TARGET].astype(int).copy()
groups = pdf[GRUPO].copy()

print(f"X: {X.shape} | features: {len(FEATURES)}")
print(f"Distribuição target:\n{y.value_counts(normalize=True).round(4)}")
print(f"Nulos:\n{X.isnull().sum()[X.isnull().sum() > 0]}")

In [0]:
# ============================================================
# CÉLULA 5 — Modelo e estratégia de validação
# Random Forest com regularização (max_depth=6, min_samples_leaf=30)
# para evitar overfitting. GroupKFold por município impede que o
# mesmo município apareça em treino e teste.
# ============================================================
modelo = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", RandomForestClassifier(n_estimators=400, max_depth=6,
                                   min_samples_leaf=30, max_features="sqrt",
                                   random_state=42, class_weight="balanced",
                                   n_jobs=-1))
])

cv = GroupKFold(n_splits=5)

In [0]:
# ============================================================
# CÉLULA 6 — Predições out-of-fold
# Cada município é previsto por um modelo que NÃO o viu no treino.
# São essas métricas que vão para o relatório — não as do treino completo.
# ============================================================
y_pred = cross_val_predict(modelo, X, y, groups=groups, cv=cv, method="predict")
y_proba = cross_val_predict(modelo, X, y, groups=groups, cv=cv, method="predict_proba")[:, 1]

print(f"ROC-AUC out-of-fold: {roc_auc_score(y, y_proba):.4f}")
print("\n=== Classification Report (out-of-fold) ===")
print(classification_report(y, y_pred, target_names=["abaixo", "atingiu"]))

In [0]:
# ============================================================
# CÉLULA 7 — Matriz de confusão
# Mostra onde o modelo erra: falsos positivos (previu "atingiu"
# e era "abaixo") e falsos negativos (o inverso).
# ============================================================
cm = confusion_matrix(y, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix=cm,
                       display_labels=["abaixo", "atingiu"]).plot(
    ax=ax, cmap="Blues", values_format="d")
ax.set_title("Matriz de Confusão — GroupKFold out-of-fold")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"\nVerdadeiros Negativos: {tn}")
print(f"Falsos Positivos:      {fp}")
print(f"Falsos Negativos:      {fn}")
print(f"Verdadeiros Positivos: {tp}")
print(f"\nTaxa de acerto: {(tn + tp) / cm.sum():.4f}")

In [0]:
# ============================================================
# CÉLULA 8 — Curva ROC
# Compara o modelo com o chute aleatório (linha diagonal).
# Quanto mais a curva sobe para o canto superior esquerdo, melhor.
# ============================================================
fpr, tpr, _ = roc_curve(y, y_proba)
auc = roc_auc_score(y, y_proba)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color="#1f77b4", lw=2, label=f"Random Forest (AUC = {auc:.4f})")
ax.plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--", label="Aleatório (AUC = 0.5000)")
ax.set_xlabel("Taxa de Falsos Positivos")
ax.set_ylabel("Taxa de Verdadeiros Positivos")
ax.set_title("Curva ROC — Validação Cruzada GroupKFold")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [0]:
# ============================================================
# CÉLULA 9 — Treinar na base completa
# Necessário para o SHAP: ele explica o modelo final, treinado
# com todos os dados. As métricas daqui NÃO vão para o relatório.
# ============================================================
modelo.fit(X, y)

X_imputado = pd.DataFrame(
    modelo.named_steps["imputer"].transform(X),
    columns=FEATURES
)
clf = modelo.named_steps["clf"]

In [0]:
explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_imputado)

# Random Forest retorna (n_amostras, n_features, n_classes) — pegamos a classe positiva
if isinstance(shap_values, list):
    shap_values = shap_values[1]
elif shap_values.ndim == 3:
    shap_values = shap_values[:, :, 1]

print(f"Shape dos SHAP values: {shap_values.shape}")

In [0]:
shap.summary_plot(shap_values, X_imputado, plot_type="dot", show=True)

In [0]:
importancia_shap = pd.DataFrame({
    "feature": FEATURES,
    "shap_medio_abs": np.abs(shap_values).mean(axis=0)
}).sort_values("shap_medio_abs", ascending=False)

display(importancia_shap)

In [0]:
# ============================================================
# CÉLULA 13 — Dependência das 4 features mais importantes
# Mostra como o SHAP varia conforme o valor da feature.
# Revela se o efeito é linear, em degrau ou tem ponto de virada.
# ============================================================
top4 = importancia_shap.head(4)["feature"].tolist()

for feat in top4:
    print(f"\n=== {feat} ===")
    shap.dependence_plot(feat, shap_values, X_imputado, show=True)

In [0]:
# ============================================================
# CÉLULA 14 — SHAP local: um município
# Explica a previsão de UM município: quais features puxaram
# para "atingiu" e quais puxaram para "abaixo".
# É a prova de interpretabilidade que o professor cobra.
# ============================================================
idx = 0
municipio = pdf.iloc[idx]

print(f"Município: {municipio['id_municipio']} | UF: {municipio['sigla_uf']}")
print(f"Target real: {'atingiu' if municipio['target'] == 1 else 'abaixo'}")
print(f"Probabilidade prevista: {modelo.predict_proba(X.iloc[[idx]])[0][1]:.4f}")
print(f"\nFeatures:")
for f in FEATURES:
    print(f"  {f:28s}: {municipio[f]}")

base_value = explainer.expected_value
if isinstance(base_value, (list, np.ndarray)):
    base_value = base_value[1]

shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[idx],
        base_values=base_value,
        data=X_imputado.iloc[idx].values,
        feature_names=FEATURES
    )
)

In [0]:
mlflow.set_experiment("/Users/rm374983@fiap.com.br/fase3_alfabetizacao")

with mlflow.start_run(run_name="random_forest_final_shap_v2"):
    mlflow.log_params({
        "modelo": "random_forest",
        "n_features": len(FEATURES),
        "cv": "GroupKFold-5",
        "max_depth": 6,
        "n_estimators": 400,
        "min_samples_leaf": 30,
    })
    mlflow.log_metric("roc_auc_oof", roc_auc_score(y, y_proba))
    mlflow.log_metric("accuracy_oof", (y_pred == y).mean())

    mlflow.log_text(importancia_shap.to_csv(index=False), "shap_importancia.csv")
    mlflow.sklearn.log_model(
        modelo,
        name="modelo_random_forest",
        input_example=X.head(5)
    )

    print("Run registrado no MLflow")